# 1. Desempeño de las ventas

**Cynthia, Antonio y Juan Diego · CRISP-DM**

Analizar generación de ventas, tendencia, contribución y desempeño descriptivo del personal para apoyar a gerencia comercial.

**NOTA: en los archivos actuales falta la tabla de transacciones. No se presentan resultados de ventas ni métricas de modelo inventadas.**

## Comprensión del negocio

**Stakeholder:** gerencia y jefatura comercial; RR. HH. para evaluación de competencias.

**Pregunta:** ¿podemos anticipar siete días de ventas mejor que repetir la semana anterior y detectar diferencias entre vendedores que ameriten revisar cartera, turnos y acompañamiento?

**Objetivo analítico:** regresión multihorizonte de ingreso diario agregado y análisis descriptivo del equipo.

**Éxito técnico:** al menos 5% de reducción relativa de WAPE frente a la línea base en validación y confirmación en los 28 días finales. MAE, RMSE y sesgo complementan la evaluación.

**Éxito comercial:** ventas recuperadas frente al costo de un piloto. NOTA: no existen resultados de un piloto ni costos para estimar ROI.

In [1]:
import json
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Image, display

from ventas.nodes import discover, paths_from_inventory

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("NOTA: el notebook requiere ejecutarse desde la raíz del proyecto.")
params = yaml.safe_load((ROOT / "conf/base/parameters.yml").read_text(encoding="utf-8"))["sales"]
inventory = discover(params)
display(inventory)
try:
    paths = paths_from_inventory(inventory)
    ready = True
except ValueError as error:
    ready = False
    print(str(error))

C:\Users\juanl\OneDrive\Documents\GitHub\Machine Learning Final\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,archivo,tabla,bytes,columnas
0,C:\Users\juanl\OneDrive\Documents\GitHub\Machi...,customers,4436738,"Address, CityID, CustomerID, FirstName, LastNa..."
1,C:\Users\juanl\OneDrive\Documents\GitHub\Machi...,countries,3179,"CountryCode, CountryID, CountryName"
2,C:\Users\juanl\OneDrive\Documents\GitHub\Machi...,products,36846,"CategoryID, Class, IsAllergic, ModifyDate, Pri..."
3,C:\Users\juanl\OneDrive\Documents\GitHub\Machi...,cities,2170,"CityID, CityName, CountryID, Zipcode"
4,C:\Users\juanl\OneDrive\Documents\GitHub\Machi...,categories,137,"CategoryID, CategoryName"
5,C:\Users\juanl\OneDrive\Documents\GitHub\Machi...,employees,1715,"BirthDate, CityID, EmployeeID, FirstName, Gend..."


NOTA: Fuentes incompletas o ambiguas. Faltantes: ['sales']; tablas repetidas: []. Consulte inventario.csv.


## Comprensión y preparación de datos

Las fuentes se identifican por columnas y se validan antes de unirlas. Se comprueban claves únicas, referencias, fechas, cantidades, descuentos y consistencia de tickets. La lectura por lotes y SQLite temporal acotan el consumo de memoria.

Los extremos válidos se conservan. Los días sin registros no se imputan como cero sin confirmación explícita. No se comparan meses incompletos como si fueran completos.

**NOTA:** la vista previa original muestra `TotalPrice` en cero. La configuración usa `Quantity × Price × (1 − Discount)`, rotulado como ingreso estimado a precio de catálogo. **NOTA:** no hay moneda ni precios históricos documentados.

In [2]:
if ready:
    product_preview = pd.read_csv(paths["products"])
    employee_preview = pd.read_csv(paths["employees"])
    display(
        pd.DataFrame(
            {
                "tabla": ["products", "employees"],
                "filas": [len(product_preview), len(employee_preview)],
            }
        )
    )
else:
    print("NOTA: no se pueden preparar ventas porque falta la tabla de transacciones.")

NOTA: no se pueden preparar ventas porque falta la tabla de transacciones.


## Modelado y evaluación

HistGradientBoostingRegressor compite contra una línea base estacional semanal. Optuna optimiza sus hiperparámetros con validación expansiva por bloques de siete días. Las variables de cada horizonte solo usan historia conocida al origen.

La elección se congela con validación. Los últimos 28 días evalúan esa decisión, actualizando entrenamiento cada semana y manteniendo hiperparámetros. Si la complejidad no se justifica, se conserva la línea base. **NOTA:** cinco meses no permiten inferir estacionalidad anual.

In [3]:
pipeline_succeeded = False
if ready:
    from kedro.framework.session import KedroSession
    from kedro.framework.startup import bootstrap_project

    bootstrap_project(ROOT)
    with KedroSession.create(project_path=ROOT) as session:
        session.run()
    pipeline_succeeded = True
else:
    print("NOTA: entrenamiento y evaluación pendientes de transacciones reales.")

NOTA: entrenamiento y evaluación pendientes de transacciones reales.


## Resultados y desempeño del personal

Se reportan generación de ventas, participación, ticket promedio, descuento ponderado, venta por día observado y tendencia de 28 días. Las diferencias son descriptivas: no prueban habilidad ni impacto individual.

**NOTA:** faltan turnos, afluencia, cuotas, costos, stock y asignación de cartera para realizar comparaciones ajustadas.

In [4]:
output = ROOT / params["output_dir"]
if pipeline_succeeded:
    quality = json.loads((output / "calidad.json").read_text(encoding="utf-8"))
    metrics = json.loads((output / "metricas_modelo.json").read_text(encoding="utf-8"))
    display(pd.Series(quality, name="Control de calidad"))
    display(pd.Series(metrics, name="Evaluación temporal"))
    display(pd.read_csv(output / "desempeno_vendedores.csv"))
    display(pd.read_csv(output / "tendencia_mensual.csv"))
    display(Image(filename=str(output / "resultados.png")))
else:
    print("NOTA: faltan resultados reales para tendencia, contribución o ranking de vendedores.")

NOTA: faltan resultados reales para tendencia, contribución o ranking de vendedores.


## Accionables y perfil para contratación

| Responsable | Acción propuesta | Validación |
|---|---|---|
| Gerencia comercial | Revisar pronóstico y capacidad semanalmente | Error y sesgo frente a línea base |
| Jefatura comercial | Investigar caídas persistentes de 28 días | Cartera, turnos, stock y oportunidades |
| Jefatura comercial | Pilotear acompañamiento entre pares comparables | Venta incremental y costo |
| RR. HH. | Usar pruebas de atención, conocimiento de productos, venta complementaria y registro | Consistencia entre evaluadores y seguimiento prospectivo |

**NOTA:** no hay datos de candidatos ni resultados prospectivos para validar un perfil predictivo de contratación. No se usan edad ni género como criterio del modelo.

Las recomendaciones son propuestas de decisión. **NOTA:** no pueden atribuirse a hallazgos del supermercado hasta disponer de las transacciones.

## Implementación reproducible

`kedro run --pipelines discovery` genera el inventario. `kedro run` ejecuta preparación, análisis, Optuna, evaluación y comunicación cuando las fuentes están completas. El catálogo y los parámetros están en `conf/base/`. GitHub Actions ejecuta Ruff, pruebas y una integración con datos sintéticos separados de las salidas reales.

**NOTA:** los outputs sintéticos verifican el software y no constituyen evidencia comercial. El diagnóstico detallado está en `docs/diagnostico_datos.md`; la decisión y sus métricas están en `docs/diseno_negocio.md`.